# Fixed human validation comparison

## 1. Setup

This notebook compares three counterattack detectors against the same 48 human-labeled validation clips: the rule-based definition, the baseline XGBoost model and the hybrid human-guided XGBoost model. The human annotations are read from the project-local Word file in Data/annotations.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from analyze_inter_rater_agreement import load_docx_rating_table

# Run this notebook from the Data_DBU project folder.
BASE_DIR = Path(".").resolve()
DOCX_PATH = BASE_DIR / "Data" / "annotations" / "Kontra vurdering (1).docx"
OUT_DIR = BASE_DIR / "Data" / "derived" / "validation_legacy_broad_pool"
AI_EVAL_DIR = BASE_DIR / "Data" / "derived" / "ai_counterattack_detection" / "human_guided_old48_eval"
SHARED_FEATURE_EVAL_DIR = BASE_DIR / "Data" / "derived" / "ai_counterattack_detection" / "shared_feature_old48_eval"

LABELS_PATH = OUT_DIR / "current_comparison_docx_labels.csv"
METRICS_PATH = OUT_DIR / "current_comparison_metrics.csv"
CONFUSION_LONG_PATH = OUT_DIR / "current_comparison_confusion_long.csv"

ALIGNED_LABELS_PATH = AI_EVAL_DIR / "old48_aligned_labels_and_predictions.csv"
BASELINE_BEST_PATH = SHARED_FEATURE_EVAL_DIR / "shared_feature_baseline_old48_predictions.csv"
HYBRID_BEST_PATH = SHARED_FEATURE_EVAL_DIR / "shared_feature_hybrid_old48_predictions.csv"

assert DOCX_PATH.exists(), f"Missing file: {DOCX_PATH}"
assert ALIGNED_LABELS_PATH.exists(), f"Missing file: {ALIGNED_LABELS_PATH}"
assert BASELINE_BEST_PATH.exists(), f"Missing file: {BASELINE_BEST_PATH}"
assert HYBRID_BEST_PATH.exists(), f"Missing file: {HYBRID_BEST_PATH}"

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Helper functions

The model outputs are stored in separate CSV files, so they are first aligned by clip_id and candidate identifiers. The evaluation function then calculates the confusion-matrix

In [2]:
def load_model_outputs():
    aligned_df = pd.read_csv(ALIGNED_LABELS_PATH)
    baseline_df = pd.read_csv(BASELINE_BEST_PATH)
    hybrid_df = pd.read_csv(HYBRID_BEST_PATH)

    for df in (aligned_df, baseline_df, hybrid_df):
        if "match_id" in df.columns:
            df["match_id"] = df["match_id"].astype(str)
        if "start_event_id" in df.columns:
            df["start_event_id"] = df["start_event_id"].astype(str)

    aligned_df = aligned_df[
        ["clip_id", "match_id", "start_event_id", "current_clip_file", "current_definition_label"]
    ].rename(columns={"current_definition_label": "definition_label"})

    baseline_df = baseline_df[
        ["match_id", "start_event_id", "pred_probability", "pred_label"]
    ].rename(columns={
        "pred_probability": "baseline_model_probability",
        "pred_label": "baseline_model_label",
    })

    hybrid_df = hybrid_df[
        ["clip_id", "pred_probability", "pred_label"]
    ].rename(columns={
        "pred_probability": "hybrid_model_probability",
        "pred_label": "hybrid_model_label",
    })

    merged = aligned_df.merge(
        baseline_df,
        on=["match_id", "start_event_id"],
        how="left",
        validate="one_to_one",
    ).merge(
        hybrid_df,
        on="clip_id",
        how="left",
        validate="one_to_one",
    )

    required_cols = ["definition_label", "baseline_model_label", "hybrid_model_label"]
    if merged[required_cols].isna().any().any():
        missing = merged.loc[merged[required_cols].isna().any(axis=1), ["clip_id"] + required_cols]
        raise ValueError(f"Missing labels after merge: {missing}")

    for col in required_cols:
        merged[col] = merged[col].astype(int)

    return merged


def safe_div(num, den):
    return num / den if den else None


def evaluate_model_vs_humans(df, model_col, model_name):
    y_true = df["human_label_num"].astype(int)
    y_pred = df[model_col].astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    accuracy = safe_div(tp + tn, len(df))
    f1 = safe_div(2 * precision * recall, precision + recall) if precision is not None and recall is not None and (precision + recall) > 0 else None

    metrics_df = pd.DataFrame([
        {
            "model": model_name,
            "n_annotated_candidates": len(df),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "accuracy": accuracy,
            "f1": f1,
        }
    ])

    confusion_df = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Human: no counterattack", "Human: counterattack"],
        columns=[f"{model_name}: no counterattack", f"{model_name}: counterattack"],
    )

    confusion_long_df = pd.DataFrame([
        {"model": model_name, "human_label": 0, "model_prediction": 0, "n": tn},
        {"model": model_name, "human_label": 0, "model_prediction": 1, "n": fp},
        {"model": model_name, "human_label": 1, "model_prediction": 0, "n": fn},
        {"model": model_name, "human_label": 1, "model_prediction": 1, "n": tp},
    ])

    return metrics_df, confusion_df, confusion_long_df

## 3. Build the comparison table

The human label is calculated by majority vote across the seven raters. Since the labels are binary and there are seven raters, this is equivalent to requiring at least four positive votes.

In [3]:
ratings_df = load_docx_rating_table(DOCX_PATH)

required_cols = ["KLIP"]
missing_cols = [col for col in required_cols if col not in ratings_df.columns]
if missing_cols:
    raise KeyError(f"Word table is missing columns: {missing_cols}")

human_cols = [col for col in ratings_df.columns if col not in ["KLIP", "Pipeline"]]
if len(human_cols) != 7:
    raise ValueError(f"Expected 7 human label columns, found {len(human_cols)}: {human_cols}")

for col in human_cols:
    ratings_df[col] = pd.to_numeric(ratings_df[col], errors="coerce")

if ratings_df[human_cols].isna().any().any():
    raise ValueError("Found missing or non-numeric human labels in the Word table.")

human_positive_votes = ratings_df[human_cols].sum(axis=1)
ratings_df["human_label_num"] = (human_positive_votes >= 4).astype(int)
ratings_df = ratings_df.rename(columns={"KLIP": "clip_id"})

model_outputs_df = load_model_outputs()
comparison_df = ratings_df.merge(model_outputs_df, on="clip_id", how="left", validate="one_to_one")

model_specs = [
    ("definition_label", "Final rule-based definition"),
    ("baseline_model_label", "XGBoost baseline model"),
    ("hybrid_model_label", "Hybrid XGBoost model"),
]

all_metrics = []
all_confusion_long = []
for model_col, model_name in model_specs:
    metrics_df, confusion_df, confusion_long_df = evaluate_model_vs_humans(comparison_df, model_col, model_name)
    all_metrics.append(metrics_df)
    all_confusion_long.append(confusion_long_df)
    print(model_name)
    print(confusion_df.to_string())
    print()

metrics_df = pd.concat(all_metrics, ignore_index=True)
confusion_long_df = pd.concat(all_confusion_long, ignore_index=True)

comparison_df.to_csv(LABELS_PATH, index=False)
metrics_df.to_csv(METRICS_PATH, index=False)
confusion_long_df.to_csv(CONFUSION_LONG_PATH, index=False)

print(f"Saved extracted labels to: {LABELS_PATH.relative_to(BASE_DIR)}")
print(f"Saved metrics to: {METRICS_PATH.relative_to(BASE_DIR)}")
print(f"Saved confusion counts to: {CONFUSION_LONG_PATH.relative_to(BASE_DIR)}")

display(comparison_df[[
    "clip_id",
    "definition_label",
    "baseline_model_label",
    "hybrid_model_label",
    "human_label_num",
] + human_cols])
display(metrics_df)

Final rule-based definition
                         Final rule-based definition: no counterattack  Final rule-based definition: counterattack
Human: no counterattack                                             17                                           6
Human: counterattack                                                 6                                          19

XGBoost baseline model
                         XGBoost baseline model: no counterattack  XGBoost baseline model: counterattack
Human: no counterattack                                        16                                      7
Human: counterattack                                            4                                     21

Hybrid XGBoost model
                         Hybrid XGBoost model: no counterattack  Hybrid XGBoost model: counterattack
Human: no counterattack                                      16                                    7
Human: counterattack                                          3 

,clip_id,definition_label,baseline_model_label,hybrid_model_label,human_label_num,Nicolai,Christoffer,Mikkel,Son,Rida,Malik,Christian
0,1,1,1,1,1,1,0,1,0,1,1,0
1,2,1,1,1,1,0,0,1,0,1,1,1
2,3,0,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,1,1,0,1,0,0,0
4,5,1,1,1,1,1,1,1,1,1,1,1
5,6,0,1,1,0,0,0,0,0,0,0,0
6,7,0,0,0,0,0,0,0,0,0,0,0
7,8,1,1,1,0,0,0,0,0,0,0,0
8,9,1,1,1,1,1,1,1,1,1,0,1
9,10,0,0,0,0,0,0,0,0,0,0,0


,model,n_annotated_candidates,tp,fp,fn,tn,precision,recall,specificity,accuracy,f1
0,Final rule-based definition,48,19,6,6,17,0.760000,0.76,0.739130,0.750000,0.760000
1,XGBoost baseline model,48,21,7,4,16,0.750000,0.84,0.695652,0.770833,0.792453
2,Hybrid XGBoost model,48,22,7,3,16,0.758621,0.88,0.695652,0.791667,0.814815
